# Training a new model in the same architecture but with less data and testing and comparing it across original model.

## Goal
Test how the pre-trained model performs against the different metrics

## Metrics
(Similiar to the classes in the Study)
- Class 6: Hand-held, Back-pocket, Front-pocket, Coat-pocket, Lower-Back, Shoulder-Bag
- Class 5: Hand-held, Pocket (Back and Front combined), Coat-pocket, Lower-Back, Shoulder-Bag
- Class 4: Hand-held, Pocket (Back, Front, and Coat combined), Lower-Back, Shoulder-Bag

(New Classes)
- MUL-SB Class 5: Hand-held, Back-pocket, Front-pocket, Coat-pocket, Lower-Back [ Unlearning Shoulder-Bag ]
- MUL-SB Class 4: Hand-held, Back-pocket (combined with Lower-Back), Front-pocket, Coat-pocket [ Unlearning Shoulder-Bag ]
- MUL-SB Class 3: Hand-held, Pocket (Lower-Back, Front-Pocket, Back-Pocket), Coat-pocket [ Unlearning Shoulder-Bag ]
- MUL-SB Class 2: Hand-held, Pocket [ Unlearning Shoulder-Bag ]

- MUL-LB Class 5: Hand-held, Back-pocket, Front-pocket, Coat-pocket, Shoulder-Bag [ Unlearning Lower-Back ]
- MUL-LB Class 4: Hand-held, Back-pocket (combined with Lower-Back), Front-pocket, Coat-pocket [ Unlearning Lower-Back ]
- MUL-LB Class 3: Hand-held, Pocket (Front-Pocket and Back-Pocket), Coat-pocket, Shoulder-Bag [ Unlearning Lower-Back ]
- MUL-LB Class 2: Hand-held, Pocket, Shoulder-Bag [ Unlearning Lower-Back ]

In [1]:
%pip install mat-io numpy pandas scipy


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import matio
import numpy as np
import pandas as pd

from model import get_model

model = get_model()
CLASS_NAMES = model.class_names
print("Class 6 labels:", CLASS_NAMES)

Loading model and feature list...
Loaded 50 selected features.
Loaded ensemble of 449 decision trees successfully.
Class 6 labels: ['LB', 'H', 'BP', 'FP', 'CP', 'SB']


In [3]:
DATA_DIR = "smartphone-placement-recognition/data"
LAB_DATA_PATH = f"{DATA_DIR}/customLab.mat"
FREELIVING_DATA_PATH = f"{DATA_DIR}/customFreeLiving.mat"

# Extra axial-correlation features present in the lab dataset only (see scripts/test_SPR.m)
LAB_CORR_COLS = [
    "CorrAccXY", "CorrAccXZ", "CorrAccYZ",
    "CorrGyrXY", "CorrGyrXZ", "CorrGyrYZ",
]


def normalize_label(value) -> str:
    if isinstance(value, (tuple, list, np.ndarray)):
        value = value[0] if len(value) else value
    return str(value).strip()


def load_dataset(path: str) -> pd.DataFrame:
    return matio.load_from_mat(path)["finalTable"]


def prepare_lab_data(df: pd.DataFrame, min_mean_gyr: float = 20) -> pd.DataFrame:
    df = df.drop(columns=[c for c in LAB_CORR_COLS if c in df.columns])
    return df[df["MeanGyr"] >= min_mean_gyr].copy()


def prepare_freeliving_data(df: pd.DataFrame, min_mean_gyr: float = 10) -> pd.DataFrame:
    return df[df["MeanGyr"] >= min_mean_gyr].copy()


def predict_class6(model, df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    features = df[model.selected_features].to_numpy(dtype=float)
    y_true = df["Position"].map(normalize_label).to_numpy()

    y_pred = []
    for row in features:
        scores = model.run_predictions(row)
        y_pred.append(max(scores, key=scores.get))

    return y_true, np.array(y_pred)

In [4]:
def evaluate_class6(y_true: np.ndarray, y_pred: np.ndarray, labels: list[str]) -> dict:
    label_to_idx = {label: i for i, label in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=int)

    for true_label, pred_label in zip(y_true, y_pred):
        cm[label_to_idx[true_label], label_to_idx[pred_label]] += 1

    accuracy = np.trace(cm) / cm.sum()
    per_class = {}
    for i, label in enumerate(labels):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        recall = tp / (tp + fn) if tp + fn else 0.0
        precision = tp / (tp + fp) if tp + fp else 0.0
        specificity = tn / (tn + fp) if tn + fp else 0.0
        per_class[label] = {
            "support": int(cm[i, :].sum()),
            "recall": recall,
            "precision": precision,
            "balanced_accuracy": (recall + specificity) / 2,
        }

    return {
        "accuracy": accuracy,
        "confusion_matrix": cm,
        "per_class": per_class,
    }


def print_class6_report(title: str, results: dict, labels: list[str]) -> None:
    print(f"\n=== {title} ===")
    print(f"Accuracy: {results['accuracy']:.4f}")

    print("\nPer-class metrics:")
    for label in labels:
        m = results["per_class"][label]
        print(
            f"  {label:>2} | support={m['support']:>4} | "
            f"recall={m['recall']:.3f} | precision={m['precision']:.3f} | "
            f"balanced_acc={m['balanced_accuracy']:.3f}"
        )

    cm = results["confusion_matrix"]
    header = " ".join(f"{label:>4}" for label in labels)
    print("\nConfusion matrix (rows=true, cols=pred):")
    print("     ", header)
    for i, label in enumerate(labels):
        row = " ".join(f"{cm[i, j]:>4}" for j in range(len(labels)))
        print(f"{label:>4} {row}")

## Test the original model against availabe dataset

### Test the original model aginst customLab.mat

In [5]:
# In-lab external validation (matches scripts/test_SPR.m preprocessing)
lab_df = prepare_lab_data(load_dataset(LAB_DATA_PATH), min_mean_gyr=20)
lab_y_true, lab_y_pred = predict_class6(model, lab_df)
lab_results = evaluate_class6(lab_y_true, lab_y_pred, CLASS_NAMES)
print_class6_report("Class 6 — customLab.mat", lab_results, CLASS_NAMES)


=== Class 6 — customLab.mat ===
Accuracy: 0.7938

Per-class metrics:
  LB | support= 430 | recall=0.865 | precision=0.756 | balanced_acc=0.906
   H | support= 463 | recall=0.914 | precision=0.828 | balanced_acc=0.937
  BP | support= 460 | recall=0.717 | precision=0.846 | balanced_acc=0.845
  FP | support= 463 | recall=0.635 | precision=0.945 | balanced_acc=0.814
  CP | support= 426 | recall=0.728 | precision=0.610 | balanced_acc=0.820
  SB | support= 459 | recall=0.904 | precision=0.849 | balanced_acc=0.936

Confusion matrix (rows=true, cols=pred):
        LB    H   BP   FP   CP   SB
  LB  372   12   23    0   23    0
   H    3  423    1    0   20   16
  BP   64    6  330    2   58    0
  FP   46   13   32  294   77    1
  CP    7   34    3   15  310   57
  SB    0   23    1    0   20  415


### Test the original model aginst customFreeLiving.mat

In [6]:
# Free-living validation (same motion threshold used during training)
fl_df = prepare_freeliving_data(load_dataset(FREELIVING_DATA_PATH), min_mean_gyr=10)
fl_y_true, fl_y_pred = predict_class6(model, fl_df)
fl_results = evaluate_class6(fl_y_true, fl_y_pred, CLASS_NAMES)
print_class6_report("Class 6 — customFreeLiving.mat", fl_results, CLASS_NAMES)


=== Class 6 — customFreeLiving.mat ===
Accuracy: 0.9130

Per-class metrics:
  LB | support=2156 | recall=0.988 | precision=0.872 | balanced_acc=0.979
   H | support=2089 | recall=0.952 | precision=0.985 | balanced_acc=0.974
  BP | support=2144 | recall=0.901 | precision=0.901 | balanced_acc=0.940
  FP | support=2144 | recall=0.724 | precision=0.974 | balanced_acc=0.860
  CP | support=2034 | recall=0.939 | precision=0.825 | balanced_acc=0.950
  SB | support=2038 | recall=0.980 | precision=0.954 | balanced_acc=0.985

Confusion matrix (rows=true, cols=pred):
        LB    H   BP   FP   CP   SB
  LB 2131    3   13    0    9    0
   H    0 1988    5    0   53   43
  BP  164    1 1931   18   30    0
  FP  139    9  164 1552  280    0
  CP    9   11   29   22 1909   54
  SB    1    6    0    2   32 1997


### Test the original model aginst customLab.mat + customFreeLiving.mat

In [7]:
# Combined dataset validation (combining both lab and free-living datasets)
combined_df = pd.concat([lab_df, fl_df], ignore_index=True)
combined_y_true, combined_y_pred = predict_class6(model, combined_df)
combined_results = evaluate_class6(combined_y_true, combined_y_pred, CLASS_NAMES)
print_class6_report("Class 6 — Combined Dataset", combined_results, CLASS_NAMES)


=== Class 6 — Combined Dataset ===
Accuracy: 0.8919

Per-class metrics:
  LB | support=2586 | recall=0.968 | precision=0.853 | balanced_acc=0.967
   H | support=2552 | recall=0.945 | precision=0.953 | balanced_acc=0.968
  BP | support=2604 | recall=0.868 | precision=0.893 | balanced_acc=0.923
  FP | support=2607 | recall=0.708 | precision=0.969 | balanced_acc=0.852
  CP | support=2460 | recall=0.902 | precision=0.787 | balanced_acc=0.928
  SB | support=2497 | recall=0.966 | precision=0.934 | balanced_acc=0.976

Confusion matrix (rows=true, cols=pred):
        LB    H   BP   FP   CP   SB
  LB 2503   15   36    0   32    0
   H    3 2411    6    0   73   59
  BP  228    7 2261   20   88    0
  FP  185   22  196 1846  357    1
  CP   16   45   32   37 2219  111
  SB    1   29    1    2   52 2412


### Results

The accuracy of the model against the provided sample dataset is representative of the claims made in the paper.

summary:

- overall: 0.89

- hand-held: 0.97 (balanced precision and recall)
- back-pocket: 0.92 (balanced precision)
- front-pocket: 0.85 (higher precision)